In [1]:
gemini_api_key='YOUR_GEMINI_KEY'
openaiapi='YOUR_OPENAI_KEY'


In [7]:
print("working!!")

## AI text generator notebook using gemini

system_prompt = '''

    You are an AI text generator. Your task is to rewrite the given human-written sentence in a typical AI-generated style.

Rules:
1. Preserve the EXACT meaning and information
2. Use formal, structured, AI-like tone
3. Use precise vocabulary
4. Avoid contractions, slang, or casual language
5. Output ONLY the rewritten sentence — nothing else, no explanation

Human sentence: {sentence}

AI version:



'''

In [4]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "YOUR_NVIDIA_KEY'
)

completion = client.chat.completions.create(
  model="meta/llama-3.2-3b-instruct",
  messages=[{"role":"user","content":"Explain ai to me"}],
  temperature=0.2,
  top_p=0.7,
  max_tokens=1024,
  stream=True
)

for chunk in completion:
  if chunk.choices and chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")



In [2]:
import pandas as pd
import time
import os
from openai import OpenAI
from tqdm import tqdm

# ============================================
# CONFIGURATION
# ============================================
NVIDIA_API_KEY = "YOUR_NVIDIA_KEY'
INPUT_FILE = "FINAL_10k.txt"
# INPUT_FILE = "10_test_sentences.txt"  # for quick testing
OUTPUT_FILE = "paired_dataset.csv"
PROGRESS_FILE = "progress.txt"
MODEL = "meta/llama-3.2-3b-instruct"

# ============================================
# CLIENT SETUP
# ============================================
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

# ============================================
# PROMPT
# ============================================
def build_prompt(human_text):
    return f"""You are an AI text generator. Rewrite the following human-written text in a typical AI-generated style.

Rules:
1. Preserve the EXACT meaning and information
2. Use formal, structured, AI-like tone
3. Use precise and neutral vocabulary
4. Avoid contractions, slang, or casual language
5. Output ONLY the rewritten text — no explanation, no preamble

Human text: {human_text}

AI version:"""

# ============================================
# LOAD PROGRESS (resume if crashed)
# ============================================
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return int(f.read().strip())
    return 0

def save_progress(index):
    with open(PROGRESS_FILE, "w") as f:
        f.write(str(index))

# ============================================
# GENERATE AI VERSION
# ============================================
def generate_ai_version(human_text, retries=3):
    for attempt in range(retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": build_prompt(human_text)}],
                temperature=0.2,
                top_p=0.7,
                max_tokens=1024,
                stream=False  # easier to handle for pipeline
            )
            return completion.choices[0].message.content.strip()

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(5)  # wait before retry

    return None  # if all retries fail

# ============================================
# MAIN PIPELINE
# ============================================
def run_pipeline():
    # Load human sentences
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        human_sentences = [
                line.strip() 
                for line in f 
                if line.strip() and line.strip() != "|"  # skip empty lines and separators
            ]

    print(f"Total sentences to process: {len(human_sentences)}")

    # Load existing progress
    start_index = load_progress()
    print(f"Resuming from index: {start_index}")

    # Load existing results if any
    if os.path.exists(OUTPUT_FILE) and start_index > 0:
        results_df = pd.read_csv(OUTPUT_FILE)
        results = results_df.to_dict('records')
    else:
        results = []

    # Process each sentence
    for i in tqdm(range(start_index, len(human_sentences))):
        human_text = human_sentences[i]

        # Generate AI version
        ai_text = generate_ai_version(human_text)

        if ai_text:
            results.append({
                "index": i,
                "human_text": human_text,
                "ai_text": ai_text
            })
        else:
            print(f"Failed at index {i} — skipping")

        # Save every 50 sentences
        if (i + 1) % 50 == 0:
            pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
            save_progress(i + 1)
            print(f"Progress saved at {i + 1}")

        # Small delay to avoid rate limits
        time.sleep(0.5)

    # Final save
    pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
    save_progress(len(human_sentences))
    print(f"\nDone! Total pairs generated: {len(results)}")
    print(f"Saved to {OUTPUT_FILE} ✅")

# ============================================
# RUN
# ============================================
if __name__ == "__main__":
    run_pipeline()

In [5]:
import pandas as pd

df = pd.read_csv("paired_dataset.csv")

print(f"Total pairs: {len(df)}")

# Check human text duplicates
human_duplicates = df['human_text'].duplicated().sum()
print(f"Duplicate human texts: {human_duplicates}")

# Check ai text duplicates
ai_duplicates = df['ai_text'].duplicated().sum()
print(f"Duplicate AI texts: {ai_duplicates}")

# Check fully duplicate rows
full_duplicates = df.duplicated(subset=['human_text', 'ai_text']).sum()
print(f"Fully duplicate pairs: {full_duplicates}")

# Unique counts
print(f"\nUnique human texts: {df['human_text'].nunique()}")
print(f"Unique AI texts:    {df['ai_text'].nunique()}")

# Final verdict
if human_duplicates == 0 and ai_duplicates == 0:
    print("\n✅ All 9975 pairs are completely unique!")
else:
    print(f"\n⚠️ Found duplicates — need cleaning!")
    # Remove duplicates
    df_clean = df.drop_duplicates(subset=['human_text'])
    print(f"After cleaning: {len(df_clean)} unique pairs remain")
    df_clean.to_csv("paired_dataset_clean_10k.csv", index=False)
    print("Saved cleaned version ✅")

In [1]:
import pandas
df = pandas.read_csv("paired_dataset_clean.csv")
print(f"Total pairs: {len(df)}")
print(df.head(3))

In [2]:
! pip install textstat wordcloud

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import textstat
from wordcloud import WordCloud

# ============================================
# LOAD DATA
# ============================================
df = pd.read_csv("paired_dataset_clean_5k.csv")
print(f"Total pairs: {len(df)}")

# ============================================
# OPTION A — PUNCTUATION ANALYSIS
# ============================================
def count_punct(text, char):
    return str(text).count(char)

df['human_questions'] = df['human_text'].apply(lambda x: count_punct(x, '?'))
df['ai_questions'] = df['ai_text'].apply(lambda x: count_punct(x, '?'))

df['human_exclaim'] = df['human_text'].apply(lambda x: count_punct(x, '!'))
df['ai_exclaim'] = df['ai_text'].apply(lambda x: count_punct(x, '!'))

df['human_comma'] = df['human_text'].apply(lambda x: count_punct(x, ','))
df['ai_comma'] = df['ai_text'].apply(lambda x: count_punct(x, ','))

df['human_semicolon'] = df['human_text'].apply(lambda x: count_punct(x, ';'))
df['ai_semicolon'] = df['ai_text'].apply(lambda x: count_punct(x, ';'))

print("\n--- PUNCTUATION ANALYSIS ---")
print(f"Question marks  → Human: {df['human_questions'].mean():.2f} | AI: {df['ai_questions'].mean():.2f}")
print(f"Exclamations    → Human: {df['human_exclaim'].mean():.2f}   | AI: {df['ai_exclaim'].mean():.2f}")
print(f"Commas          → Human: {df['human_comma'].mean():.2f}  | AI: {df['ai_comma'].mean():.2f}")
print(f"Semicolons      → Human: {df['human_semicolon'].mean():.2f}   | AI: {df['ai_semicolon'].mean():.2f}")

# ============================================
# OPTION B — SENTENCE LENGTH VARIANCE
# ============================================
def sentence_length_variance(text):
    sentences = [s.strip() for s in str(text).split('.') if s.strip()]
    if len(sentences) < 2:
        return 0
    lengths = [len(s.split()) for s in sentences]
    return np.var(lengths)

df['human_sent_variance'] = df['human_text'].apply(sentence_length_variance)
df['ai_sent_variance'] = df['ai_text'].apply(sentence_length_variance)

print("\n--- SENTENCE LENGTH VARIANCE (burstiness) ---")
print(f"Human variance: {df['human_sent_variance'].mean():.2f}")
print(f"AI variance:    {df['ai_sent_variance'].mean():.2f}")

# ============================================
# OPTION C — MOST COMMON WORDS
# ============================================
def get_top_words(texts, n=20):
    all_words = []
    stopwords = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on',
                     'at', 'to', 'for', 'of', 'with', 'by', 'from', 'is',
                     'it', 'this', 'that', 'was', 'are', 'be', 'as', 'has',
                     'have', 'had', 'not', 'they', 'their', 'which', 'its',
                     'also', 'been', 'were', 'would', 'could', 'should',
                     'will', 'can', 'may', 'more', 'than', 'such', 'these'])
    for text in texts:
        words = re.findall(r'\b[a-z]+\b', str(text).lower())
        filtered = [w for w in words if w not in stopwords and len(w) > 3]
        all_words.extend(filtered)
    return Counter(all_words).most_common(n)

human_top = get_top_words(df['human_text'])
ai_top = get_top_words(df['ai_text'])

print("\n--- TOP 20 WORDS ---")
print("Human:", [w[0] for w in human_top])
print("AI:   ", [w[0] for w in ai_top])

# ============================================
# OPTION D — READABILITY SCORES
# ============================================
# Install if needed: pip install textstat
def safe_flesch(text):
    try:
        return textstat.flesch_reading_ease(str(text))
    except:
        return None

def safe_grade(text):
    try:
        return textstat.flesch_kincaid_grade(str(text))
    except:
        return None

df['human_flesch'] = df['human_text'].apply(safe_flesch)
df['ai_flesch'] = df['ai_text'].apply(safe_flesch)
df['human_grade'] = df['human_text'].apply(safe_grade)
df['ai_grade'] = df['ai_text'].apply(safe_grade)

print("\n--- READABILITY (Flesch Reading Ease) ---")
print("Higher = easier to read")
print(f"Human: {df['human_flesch'].mean():.2f}")
print(f"AI:    {df['ai_flesch'].mean():.2f}")

print("\n--- GRADE LEVEL (Flesch-Kincaid) ---")
print("Higher = harder to read")
print(f"Human: {df['human_grade'].mean():.2f}")
print(f"AI:    {df['ai_grade'].mean():.2f}")

# ============================================
# VISUALIZATIONS
# ============================================
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle('Deep Analysis: Human vs AI Text', fontsize=18, fontweight='bold')

# Plot 1 — Punctuation comparison
punct_metrics = ['Questions (?)', 'Exclamations (!)', 'Commas (,)', 'Semicolons (;)']
human_punct = [df['human_questions'].mean(), df['human_exclaim'].mean(),
               df['human_comma'].mean(), df['human_semicolon'].mean()]
ai_punct = [df['ai_questions'].mean(), df['ai_exclaim'].mean(),
            df['ai_comma'].mean(), df['ai_semicolon'].mean()]

x = np.arange(len(punct_metrics))
width = 0.35
axes[0,0].bar(x - width/2, human_punct, width, label='Human', color='steelblue', alpha=0.8)
axes[0,0].bar(x + width/2, ai_punct, width, label='AI', color='coral', alpha=0.8)
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(punct_metrics, fontsize=9)
axes[0,0].set_title('Punctuation Usage')
axes[0,0].legend()

# Plot 2 — Sentence variance (burstiness)
axes[0,1].hist(df['human_sent_variance'], bins=50, alpha=0.6, color='steelblue', label='Human')
axes[0,1].hist(df['ai_sent_variance'], bins=50, alpha=0.6, color='coral', label='AI')
axes[0,1].set_title('Sentence Length Variance (Burstiness)')
axes[0,1].set_xlabel('Variance')
axes[0,1].legend()

# Plot 3 — Readability
categories = ['Flesch Ease\n(higher=easier)', 'Grade Level\n(higher=harder)']
human_read = [df['human_flesch'].mean(), df['human_grade'].mean()]
ai_read = [df['ai_flesch'].mean(), df['ai_grade'].mean()]
x2 = np.arange(len(categories))
axes[1,0].bar(x2 - width/2, human_read, width, label='Human', color='steelblue', alpha=0.8)
axes[1,0].bar(x2 + width/2, ai_read, width, label='AI', color='coral', alpha=0.8)
axes[1,0].set_xticks(x2)
axes[1,0].set_xticklabels(categories)
axes[1,0].set_title('Readability Scores')
axes[1,0].legend()

# Plot 4 — Top words bar chart
human_words = [w[0] for w in human_top[:10]]
human_counts = [w[1] for w in human_top[:10]]
ai_words = [w[0] for w in ai_top[:10]]
ai_counts = [w[1] for w in ai_top[:10]]

axes[1,1].barh(human_words, human_counts, color='steelblue', alpha=0.8)
axes[1,1].set_title('Top 10 Human Words')
axes[1,1].invert_yaxis()

# Plot 5 — WordCloud Human
human_text_combined = ' '.join(df['human_text'].astype(str))
wc_human = WordCloud(width=800, height=400, background_color='white',
                     colormap='Blues').generate(human_text_combined)
axes[2,0].imshow(wc_human, interpolation='bilinear')
axes[2,0].axis('off')
axes[2,0].set_title('Human Text WordCloud')

# Plot 6 — WordCloud AI
ai_text_combined = ' '.join(df['ai_text'].astype(str))
wc_ai = WordCloud(width=800, height=400, background_color='white',
                  colormap='Oranges').generate(ai_text_combined)
axes[2,1].imshow(wc_ai, interpolation='bilinear')
axes[2,1].axis('off')
axes[2,1].set_title('AI Text WordCloud')

plt.tight_layout()
plt.savefig('deep_analysis_5k.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved to deep_analysis_5k.png ✅")